In [2]:
import pandas as pd

df = pd.read_csv("device_events.csv")
df.head()

,event_id,device_id,event_time,event_type,temperature
0,E001,D101,2026-09-22 08:00:00,START,42.5
1,E002,D102,2026-09-22 08:15:00,START,40.2
2,E003,D101,2026-09-22 09:30:00,ERROR,78.5
3,E004,D103,2026-09-22 10:00:00,START,39.8
4,E005,D102,2026-09-22 11:45:00,ERROR,82.1


In [3]:
df.shape

(12, 5)

In [4]:
df.dtypes


event_id           str
device_id          str
event_time         str
event_type         str
temperature    float64
dtype: object

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   event_id     12 non-null     str    
 1   device_id    12 non-null     str    
 2   event_time   12 non-null     str    
 3   event_type   12 non-null     str    
 4   temperature  12 non-null     float64
dtypes: float64(1), str(4)
memory usage: 991.0 bytes


In [6]:
df["event_time"] = pd.to_datetime(df["event_time"],errors = "coerce")
df.dtypes

event_id                  str
device_id                 str
event_time     datetime64[us]
event_type                str
temperature           float64
dtype: object

In [7]:
#dispaly the record containing invalid timestamp
df[df["event_time"].isna()]

,event_id,device_id,event_time,event_type,temperature
6,E007,D103,NaT,ERROR,76.8


In [8]:
df["event_date"] = df["event_time"].dt.date

In [9]:
df["event_hour"] = df["event_time"].dt.hour
df

,event_id,device_id,event_time,event_type,temperature,event_date,event_hour
0,E001,D101,2026-09-22 08:00:00,START,42.5,2026-09-22,8.0
1,E002,D102,2026-09-22 08:15:00,START,40.2,2026-09-22,8.0
2,E003,D101,2026-09-22 09:30:00,ERROR,78.5,2026-09-22,9.0
3,E004,D103,2026-09-22 10:00:00,START,39.8,2026-09-22,10.0
4,E005,D102,2026-09-22 11:45:00,ERROR,82.1,2026-09-22,11.0
5,E006,D101,2026-09-22 12:00:00,STOP,55.4,2026-09-22,12.0
6,E007,D103,NaT,ERROR,76.8,NaT,NaN
7,E008,D104,2026-09-22 13:30:00,START,41.3,2026-09-22,13.0
8,E009,D102,2026-09-22 14:15:00,STOP,51.7,2026-09-22,14.0
9,E010,D104,2026-09-22 15:00:00,ERROR,80.4,2026-09-22,15.0


In [10]:
start_time = pd.to_datetime("2026-09-22 00:00:00")
end_time = pd.to_datetime("2026-09-22 23:59:59")

daily_processing_df = df[(df["event_time"] >= start_time) & (df["event_time"] <= end_time)]
daily_processing_df

,event_id,device_id,event_time,event_type,temperature,event_date,event_hour
0,E001,D101,2026-09-22 08:00:00,START,42.5,2026-09-22,8.0
1,E002,D102,2026-09-22 08:15:00,START,40.2,2026-09-22,8.0
2,E003,D101,2026-09-22 09:30:00,ERROR,78.5,2026-09-22,9.0
3,E004,D103,2026-09-22 10:00:00,START,39.8,2026-09-22,10.0
4,E005,D102,2026-09-22 11:45:00,ERROR,82.1,2026-09-22,11.0
5,E006,D101,2026-09-22 12:00:00,STOP,55.4,2026-09-22,12.0
7,E008,D104,2026-09-22 13:30:00,START,41.3,2026-09-22,13.0
8,E009,D102,2026-09-22 14:15:00,STOP,51.7,2026-09-22,14.0
9,E010,D104,2026-09-22 15:00:00,ERROR,80.4,2026-09-22,15.0
10,E011,D103,2026-09-22 16:20:00,STOP,49.9,2026-09-22,16.0


In [19]:
daily_processing_df[daily_processing_df["event_type"] == "ERROR"].sort_values(by="temperature", ascending=False)

,event_id,device_id,event_time,event_type,temperature,event_date,event_hour
4,E005,D102,2026-09-22 11:45:00,ERROR,82.1,2026-09-22,11.0
9,E010,D104,2026-09-22 15:00:00,ERROR,80.4,2026-09-22,15.0
2,E003,D101,2026-09-22 09:30:00,ERROR,78.5,2026-09-22,9.0


In [18]:
daily_processing_df.groupby("device_id").agg(
    total_event = ("event_type","count"),
    avg_temperature = ("temperature","mean"),
    max_temperature = ("temperature","max")
).reset_index()

,device_id,total_event,avg_temperature,max_temperature
0,D101,3,58.80,78.5
1,D102,3,58.00,82.1
2,D103,2,44.85,49.9
3,D104,2,60.85,80.4


In [ ]:
daily_processing_df.groupby("event_type").agg(
    event_count = ("event_type","count"),
    avg_temperature = ("temperature","mean")
)